In [12]:
import polars as pl

# Import power plants
plants_df = pl.read_csv('Distances - Power Plants.csv')

# Import towns
towns_df = pl.read_csv('Distances - Towns.csv')

In [13]:
import math

# Formula to get the distance between two locations from the longitutude and latitutude in km
def haversine(lat1, long1, lat2, long2):
    return 2*6371*math.asin(
        math.sqrt(
            (math.sin(lat2-lat1)/2)**2)+
            math.cos(lat1)*math.cos(lat2)*math.sin((long2-long1)/2)**2)

In [14]:
rows = [] # List of (plant, town, distance) tuples

# Iterate through power plants and town pairs
for plant_row in plants_df.iter_rows(named=True):
    for town_row in towns_df.iter_rows(named=True):
        distance = haversine(plant_row["Latitude"], plant_row["Longitude"], town_row["Latitude"], town_row["Longitude"])
        rows.append((plant_row["Power Plant"], town_row["Towns"], distance))

In [15]:
columns = ["power_plant", "town", "distance"]
rows_df = pl.DataFrame(rows, schema=columns, orient="row")
rows_df.head()

power_plant,town,distance
str,str,f64
"""Essential Power Newington""","""Manchester""",1154.767881
"""Essential Power Newington""","""Nashua""",2434.719413
"""Essential Power Newington""","""Concord""",2470.095834
"""Essential Power Newington""","""Portsmouth""",192.860287
"""Essential Power Newington""","""Rochester""",1314.450331


In [16]:
# Add ID's for power plants and towns
rows_df = rows_df.with_columns(
    pl.col("power_plant").rank("dense").alias("power_plant_id"),
    pl.col("town").rank("dense").alias("town_id")
)
rows_df.head()

power_plant,town,distance,power_plant_id,town_id
str,str,f64,u32,u32
"""Essential Power Newington""","""Manchester""",1154.767881,2,6
"""Essential Power Newington""","""Nashua""",2434.719413,2,7
"""Essential Power Newington""","""Concord""",2470.095834,2,2
"""Essential Power Newington""","""Portsmouth""",192.860287,2,8
"""Essential Power Newington""","""Rochester""",1314.450331,2,9


In [18]:
rows_df.write_csv("distance_pairs.csv")